In [ ]:
import os, copy
import glob

import numpy as np
import scipy.optimize as so
import pandas as pd
import xarray as xr
import rioxarray as rxr

import netCDF4
import h5py
from osgeo import gdal

#%matplotlib inline  
import matplotlib
import matplotlib.pyplot as plt
import colorcet as cc

from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
pn.extension()
from holoviews import opts

import panel as pn
import importlib
import hgrs.driver as driver
importlib.reload(driver)


import hgrs

opj = os.path.join

hgrs.__version__

Set file paths, give path for L1C (L2C is normally in the same directory). The L2C is needed to get the viewing geometry parameters (viewing zenith angle, sun zenith angle and relative azimuth)


In [ ]:

l1c_path = '/home/zobesn/data/satellite/PRISMA/Albufera/PRS_L1_STD_OFFL_20240723110233_20240723110237_0001.he5'


l1c = os.path.basename(l1c_path)
l2c_path = l1c_path.replace('L1_STD_OFFL', 'L2C_STD')

In [ ]:
#
dc_l1c = driver.read_L1C_data(l1c_path,reflectance_unit=True)


In [ ]:
dc_l2c = driver.read_L2C_data(l2c_path)


Merge the two rasters (keep only parameters of interest from l2c)

In [ ]:
for param in ['sza', 'vza', 'raa']:
    dc_l1c[param] = dc_l2c[param]
del dc_l2c

In [ ]:
dc_l1c.fwhm

In [ ]:
mean_spectrum = dc_l1c.Ltoa.mean(dim=["x", "y"], skipna=True)
plt.figure(figsize=(5, 3))
plt.plot(mean_spectrum.wl, mean_spectrum, color='blue', linewidth=1)
plt.title("Mean spectrum of Ltoa ")
plt.xlabel("Wavelength (nm)")
plt.xlim(left=407)
plt.ylabel("TOA Radiance")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mean_spectrum = dc_l1c.Rtoa.mean(dim=["x", "y"], skipna=True)
plt.figure(figsize=(5, 3))
plt.plot(mean_spectrum.wl, mean_spectrum, color='blue', linewidth=1)
plt.title("Mean spectrum of Rtoa ")
plt.xlabel("Wavelength (nm)")
plt.ylabel("TOA Reflectance")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
dc_l1c.Ltoa.sel(wl=[440,550,660,770,880,1600],method='nearest').plot.imshow(col='wl',col_wrap=3,cmap=plt.cm.Spectral_r,robust=True)


In [ ]:
dc_l1c.Rtoa.sel(wl=[440,550,660,770,880,1600],method='nearest').plot.imshow(col='wl',col_wrap=3,cmap=plt.cm.Spectral_r,robust=True, vmin =0, vmax=0.5)


In [ ]:
dc_l1c.Rtoa.sel(wl=[440,550,660,770,880,1600],method='nearest').plot.imshow(col='wl',col_wrap=3,cmap=plt.cm.Spectral_r,robust=True, vmin =0, vmax=0.5)

## Plot and interact

In [ ]:
from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
from holoviews import opts

opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))
# set the parameter for spectra extraction
hv.extension('bokeh')
pn.extension()

raster = dc_l1c.Rtoa#.reset_coords()#.isel(time=-1,drop=True)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= 'RdBu_r',colorbar=True)#.hist(bin_range=(0,0.02) ) 
widget = pn.widgets.RangeSlider(start=0, end=1,step=0.001)

jscode = """
    color_mapper.low = cb_obj.value[0];
    color_mapper.high = cb_obj.value[1];
"""
link = widget.jslink(im, code={'value': jscode})
pn.Column(widget, im)

In [ ]:
xmode Minimal

In [ ]:


param = 'Rtoa'
raster = dc_l1c[param] 


#param = 'rho'
#raster = dc_l2c[param] 
cmap='Spectral_r'
#cmap='RdBu_r'
third_dim = 'wl'

wl= raster.wl.data
Nwl = len(wl)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= cmap,colorbar=True,clim=(0,0.315)).hist(bin_range=(0,0.02)) 

polys = hv.Polygons([])
box_stream = hv.streams.BoxEdit(source=polys)
dmap, dmap_std=[],[]

def roi_curves(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]= hv.Curve((wl,mean[param]),'Wavelength (nm)', param) 

    return hv.NdOverlay(curves)


# a bit dirty to have two similar function, but holoviews does not like mixing Curve and Spread for the same stream
def roi_spreads(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]=  hv.Spread((wl,mean[param],std[param]))
        #curves[i].opts(fill_alpha=0.3)
    return hv.NdOverlay(curves)

mean=hv.DynamicMap(roi_curves,streams=[box_stream])
std =hv.DynamicMap(roi_spreads, streams=[box_stream])    
hlines = hv.HoloMap({wl[i]: hv.VLine(wl[i]) for i in range(Nwl)},third_dim )


hv.output(widget_location='top_left')

# visualize and play
graphs = ((mean* std *hlines).relabel(param))
layout = (im * polys +graphs    ).opts(
    opts.Curve(width=750,height=500, framewise=True,xlim=(400,2500)), 
    opts.Polygons(fill_alpha=0.2, color='green',line_color='black'), 
    opts.VLine(color='black')).cols(2)
layout 

## Example of exploiation: compute NDWI for water pixel masking


In [ ]:
# Compute NDWI
green = dc_l1c.Rtoa.sel(wl=565,method='nearest')
nir = dc_l1c.Rtoa.sel(wl=1610,method='nearest')
ndwi = (green - nir) / (green + nir)

In [ ]:
coarsening=1
import matplotlib as mpl
# binary cmap
bcmap = mpl.colors.ListedColormap(['khaki', 'lightblue'])

def water_mask(ndwi, threshold=0):
    water = xr.where(ndwi > threshold, 1, 0)
    return water.where(~np.isnan(ndwi))

def plot_water_mask(ndwi,ax,threshold=0):
    water = water_mask(ndwi, threshold)
    #ax.set_extent(extent_val, proj)
    water.plot.imshow( cmap=bcmap,
                                  cbar_kwargs={'ticks': [0, 1], 'shrink': shrink})#extent=extent_val, transform=proj,
    ax.set_title(str(threshold)+' < NDWI')
    
fig = plt.figure(figsize=(20, 15))
fig.subplots_adjust(bottom=0.1, top=0.95, left=0.1, right=0.99,
                    hspace=0.05, wspace=0.05)
shrink = 0.8
    
ax = plt.subplot(2, 2, 1)#, projection=proj)
#ax.set_extent(extent_val, proj)
fig = ndwi[::coarsening, ::coarsening].plot.imshow(cmap=plt.cm.BrBG, robust=True,
                                   cbar_kwargs={'shrink': shrink})# extent=extent_val, transform=proj, 
# axes.coastlines(resolution='10m',linewidth=1)
ax.set_title('Sentinel 2, NDWI')

for i,threshold in enumerate([-0.2,0.,0.2]):
    ax = plt.subplot(2, 2, i+2)#, projection=proj)
    plot_water_mask(ndwi[::coarsening, ::coarsening],ax,threshold=threshold)

plt.show()



In [ ]:
from pyproj import CRS
import cartopy.crs as ccrs

crs = CRS.from_epsg(32630)
dc_l1c.rio.write_crs(crs, inplace=True)


str_epsg = str(dc_l1c.rio.crs)
zone = str_epsg[-2:]
is_south = str_epsg[2] == 7
proj = ccrs.UTM(zone, is_south)

In [ ]:
print(dc_l1c.acquisition_date)
time =  pd.to_datetime(dc_l1c.acquisition_date)

dc_l1c = dc_l1c.assign_coords(time=time)
print(dc_l1c.coords)

In [ ]:
mask = water_mask(ndwi, threshold=0)
data_water = dc_l1c.Rtoa.where(mask == 1)


single_band = data_water.sel(wl=622, method='nearest')

plt.figure(figsize=(5, 5))
plt.imshow(single_band[::coarsening, ::coarsening], cmap=plt.cm.Spectral_r, vmin=np.nanpercentile(single_band, 2), vmax=np.nanpercentile(single_band, 98))
plt.title("Rtoa Albufera EnMAP– 550 nm")
plt.axis('off')
plt.colorbar(label='Réflectance')
plt.show()


In [ ]:
import holoviews as hv
import panel as pn

hv.extension('bokeh')
pn.extension()
from grstbx import visual

v=visual.ViewSpectral(data_water.isel(wl=range(0,224,10))  ,reproject=True)
v.minmax=[0,0.15]
v.minmaxvalues=(0,0.15)
v.visu()

In [ ]:
geom_ = v.get_geom(v.aoi_stream,crs=crs)
raster_clipped=dc_l1c.rio.clip(geom_.geometry.values)
raster_clipped.load()

In [ ]:
 #Compute NDWI
green = raster_clipped.Rtoa.sel(wl=565, method='nearest')
nir = raster_clipped.Rtoa.sel(wl=1610, method='nearest')
ndwi = (green - nir) / (green + nir)
coarsening = 1
import matplotlib as mpl

# binary cmap
bcmap = mpl.colors.ListedColormap(['khaki', 'lightblue'])


def water_mask(ndwi, threshold=0):
    water = xr.where(ndwi > threshold, 1, 0)
    return water.where(~np.isnan(ndwi))


def plot_water_mask(ndwi, ax, threshold=0):
    water = water_mask(ndwi, threshold)
    #ax.set_extent(extent_val, proj)
    water.plot.imshow(cmap=bcmap,
                      cbar_kwargs={'ticks': [0, 1], 'shrink': shrink})  #extent=extent_val, transform=proj,
    ax.set_title(str(threshold) + ' < NDWI')


fig = plt.figure(figsize=(20, 15))
fig.subplots_adjust(bottom=0.1, top=0.95, left=0.1, right=0.99,
                    hspace=0.05, wspace=0.05)
shrink = 0.8

ax = plt.subplot(2, 2, 1)  #, projection=proj)
#ax.set_extent(extent_val, proj)
fig = ndwi[::coarsening, ::coarsening].plot.imshow(cmap=plt.cm.BrBG, robust=True,
                                                   cbar_kwargs={
                                                       'shrink': shrink})  # extent=extent_val, transform=proj,
# axes.coastlines(resolution='10m',linewidth=1)
ax.set_title('Sentinel 2, NDWI')

for i, threshold in enumerate([-0.2, 0., 0.2]):
    ax = plt.subplot(2, 2, i + 2)  #, projection=proj)
    plot_water_mask(ndwi[::coarsening, ::coarsening], ax, threshold=threshold)

plt.show()


In [ ]:
mask = water_mask(ndwi, threshold=0)
data_water = raster_clipped.Rtoa.where(mask == 1)

single_band = data_water.sel(wl=550, method='nearest')

plt.figure(figsize=(5, 5))
plt.imshow(single_band[::coarsening, ::coarsening], cmap=plt.cm.Spectral_r, vmin=0.13,
           vmax=np.nanpercentile(single_band, 98))
plt.title("Rtoa Albufera EnMAP– 550 nm")
plt.colorbar(label='Réflectance')
plt.gca().invert_xaxis()


plt.show()

